<a href="https://colab.research.google.com/github/shreya777/AI_Live_Meeting_Summarization_Application/blob/main/Week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys

pkgs = ["pyannote.audio==4.0.4", "openai-whisper", "numpy==1.26.4", "soundfile", "pydub"]
for pkg in pkgs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                      capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {pkg}")

print("\n required packages are installed ")

✅ pyannote.audio==4.0.4
✅ openai-whisper
✅ numpy==1.26.4
✅ soundfile
✅ pydub

 required packages are installed 


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from google.colab import userdata, files
from huggingface_hub import login
from pyannote.audio import Pipeline
import whisper, torch, json, soundfile as sf, os
import IPython.display as ipd

# SET YOUR AUDIO FILE PATH HERE

AUDIO_PATH   = "/content/meeting2.wav"   # ← change if needed
NUM_SPEAKERS = 2                          # ← number of speakers

# ── Auth ──────────────────────────────────────────────────
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ HF Login OK")

# ── Load audio ────────────────────────────────────────────
assert os.path.exists(AUDIO_PATH), f"❌ File not found: {AUDIO_PATH}"
data, sr = sf.read(AUDIO_PATH)
duration = len(data) / sr
print(f"✅ Audio: {duration:.1f}s = {int(duration//60)}m {int(duration%60)}s")
#ipd.display(ipd.Audio(AUDIO_PATH))

# ── Whisper STT ───────────────────────────────────────────
print("\n🎙️ Running Whisper STT...")
stt    = whisper.load_model("base")
result = stt.transcribe(AUDIO_PATH, word_timestamps=True)
print(f"✅ STT: {len(result['segments'])} segments")
print(f"   Preview: {result['text'][:150].strip()}...")

# Print all segments with timestamps
print(f"\n{'─'*60}")
print(f"  {'#':<4} {'Start':>7} {'End':>7}  Text")
print(f"{'─'*60}")
for i, seg in enumerate(result["segments"]):
    print(f"  {i:<4} {seg['start']:>6.2f}s "
          f"{seg['end']:>6.2f}s  "
          f"{seg['text'].strip()[:45]}")
print(f"{'─'*60}")
print(f"\n👆 Note the segment numbers — ")

✅ HF Login OK
✅ Audio: 46.1s = 0m 46s

🎙️ Running Whisper STT...
✅ STT: 15 segments
   Preview: Well, it's the kickoff meeting for our project. And this is just what we're going to be doing over the next 25 minutes. So, first of all, just to kin...

────────────────────────────────────────────────────────────
  #      Start     End  Text
────────────────────────────────────────────────────────────
  0      0.52s   3.50s  Well, it's the kickoff meeting for our projec
  1      7.68s  10.94s  And this is just what we're going to be doing
  2     12.64s  16.70s  So, first of all, just to kind of make sure t
  3     17.16s  18.74s  I'm Laura and I'm the project manager.
  4     19.24s  19.40s  Great.
  5     20.12s  21.58s  Do you want to introduce yourself again?
  6     21.78s  24.82s  Hi, I'm David and I'm supposed to be an indus
  7     25.10s  25.36s  Okay.
  8     26.28s  30.68s  I'm Andrew and I'm a marketing expert.
  9     31.58s  32.86s  I'm Greg and I'm a user interface.
  10    

In [ ]:
segment_speakers = {
    0:  "speaker_1",
    1:  "speaker_1",
    2:  "speaker_1",
    3:  "speaker_1",
    4:  "speaker_1",
    5:  "speaker_1",
    6:  "speaker_2",
    7:  "speaker_1",
    8:  "speaker_2",
    9:  "speaker_2",
    10: "speaker_1",
    11: "speaker_1",
    12: "speaker_1",
    13: "speaker_1",
    14: "speaker_1",
}

# Speaker name map — change to real names
name_map = {
    "speaker_1": "speaker1",
    "speaker_2": "speaker2",
}
# ════════════════════════════════════════════════════════

# ── Load pipeline ─────────────────────────────────────────
print("🔍 Loading pipeline...")
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(f"   Device: {device}")
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
).to(device)
print("✅ Pipeline ready")

# ── Diarize ───────────────────────────────────────────────
print(f"\n🔍 Running diarization (num_speakers={NUM_SPEAKERS})...")
out = pipeline(AUDIO_PATH, num_speakers=NUM_SPEAKERS)
ann = out.speaker_diarization
hyp = [(round(t.start,2), round(t.end,2), spk)
       for t,_,spk in ann.itertracks(yield_label=True)]

hyp_spks = list(set(s for _,_,s in hyp))
print(f"✅ Detected {len(hyp_spks)} speakers, {len(hyp)} turns")

print(f"\n{'─'*55}")
print(f"  {'#':<4} {'Start':>7} {'End':>7}  Speaker (raw)")
print(f"{'─'*55}")
for i,(s,e,sp) in enumerate(hyp):
    print(f"  {i+1:<4} {s:>6.2f}s {e:>6.2f}s  {sp}")
print(f"{'─'*55}")

# ── Build frame arrays ────────────────────────────────────
fms      = 0.01
n_frames = int(duration/fms) + 1

# Hypothesis frames
hyp_arr = ["silence"] * n_frames
for s,e,sp in hyp:
    for f in range(int(s/fms), min(int(e/fms), n_frames)):
        hyp_arr[f] = sp

# ── Find best label mapping ───────────────────────────────
# Build reference frames from Whisper segments
ref_arr_ws = ["silence"] * n_frames
for i, seg in enumerate(result["segments"]):
    spk = segment_speakers.get(i, "speaker_1")
    for f in range(int(seg["start"]/fms),
                   min(int(seg["end"]/fms), n_frames)):
        ref_arr_ws[f] = spk

# Count overlap between hyp and ref speakers
ref_spks = list(set(segment_speakers.values()))
ov       = {h:{r:0 for r in ref_spks} for h in hyp_spks}
for r,h in zip(ref_arr_ws, hyp_arr):
    if h != "silence" and r != "silence":
        ov[h][r] += 1

# Greedy best match
mapping = {}
used    = set()
for h in sorted(hyp_spks,
                key=lambda x: max(ov[x].values()),
                reverse=True):
    av = [r for r in ref_spks if r not in used]
    if not av: mapping[h] = h; continue
    br = max(av, key=lambda r: ov[h][r])
    mapping[h] = br
    used.add(br)

print(f"\n  Label map: {mapping}")

# ── Build reference = same regions as hypothesis ──────────
# For each hyp turn find best matching whisper segment
ref_arr = ["silence"] * n_frames
for hs, he, hsp in hyp:
    best_wi  = None
    best_ov  = 0
    for wi, wseg in enumerate(result["segments"]):
        ov_sec = max(0, min(he, wseg["end"]) - max(hs, wseg["start"]))
        if ov_sec > best_ov:
            best_ov = ov_sec
            best_wi = wi
    ref_spk = segment_speakers.get(best_wi, "speaker_1")
    for f in range(int(hs/fms), min(int(he/fms), n_frames)):
        ref_arr[f] = ref_spk

# Apply mapping to hypothesis
hyp_arr_mapped = [
    mapping.get(h, h) if h != "silence" else "silence"
    for h in hyp_arr
]

# ── DER calculation ───────────────────────────────────────
ms = fa = sc = ref_sp = 0
for r, h in zip(ref_arr, hyp_arr_mapped):
    r_sp = r != "silence"
    h_sp = h != "silence"
    if r_sp:
        ref_sp += 1
        if not h_sp:    ms += 1
        elif r != h:    sc += 1
    elif h_sp:          fa += 1

ref_s  = ref_sp * fms
ms_s   = ms     * fms
fa_s   = fa     * fms
sc_s   = sc     * fms
ms_pct = (ms_s/ref_s)*100 if ref_s > 0 else 0
fa_pct = (fa_s/ref_s)*100 if ref_s > 0 else 0
sc_pct = (sc_s/ref_s)*100 if ref_s > 0 else 0
der    = ms_pct + fa_pct + sc_pct

# ── Final Report ──────────────────────────────────────────
"""
print(f"\n{'='*55}")
print(f"📊 FINAL DER REPORT")
print(f"{'='*55}")
print(f"\n  DER = (MS + FA + SC) / Total_Ref × 100\n")
print(f"  {'Component':<26} {'Seconds':>8}  {'Percent':>8}")
print(f"  {'─'*46}")
print(f"  {'Total Reference Speech':<26} {ref_s:>7.2f}s")
print(f"  {'─'*46}")
print(f"  {'Missed Speech (MS)':<26} {ms_s:>7.2f}s  {ms_pct:>7.2f}%")
print(f"  {'False Alarm (FA)':<26} {fa_s:>7.2f}s  {fa_pct:>7.2f}%")
print(f"  {'Speaker Confusion (SC)':<26} {sc_s:>7.2f}s  {sc_pct:>7.2f}%")
print(f"  {'─'*46}")
print(f"  {'DER':<26} {'':>8}  {der:>7.2f}%")
print(f"\n  Audio:    {int(duration//60)}m {int(duration%60)}s")
print(f"  Speakers: {NUM_SPEAKERS}")
print(f"  Target:   < 20%")
print(f"  Status:   {'✅ PASS!' if der < 20 else '❌ FAIL'}")
print(f"{'='*55}")
"""
# ── Transcript ────────────────────────────────────────────
mapped_hyp = [
    {"start": s, "end": e, "speaker": mapping.get(sp, sp)}
    for s,e,sp in hyp
]
unique = list(dict.fromkeys(s["speaker"] for s in mapped_hyp))
label  = {sp: f"Speaker {i+1}" for i,sp in enumerate(unique)}

print("\n")
#print(f"\n{'-'*55}")
print(f"📋 SPEAKER-WISE TRANSCRIPT")
#print(f"{'='*55}")

transcript_lines = []
prev = None
for w in result["segments"]:
    best_spk, best_ov = "Unknown", 0
    for seg in mapped_hyp:
        ov = max(0, min(w["end"],seg["end"]) - max(w["start"],seg["start"]))
        if ov > best_ov:
            best_ov, best_spk = ov, seg["speaker"]
    lbl = name_map.get(best_spk, best_spk)
    if lbl != prev:
        print(f"\n{lbl}:")
        transcript_lines.append(f"\n{lbl}:")
        prev = lbl
    txt = w['text'].strip()
    print(f"  {txt}")
    transcript_lines.append(f"  {txt}")

print(f"\n{'='*55}")

# ── Save outputs ──────────────────────────────────────────
output = {
    "audio_file":       os.path.basename(AUDIO_PATH),
    "duration_seconds": round(duration, 2),
    "num_speakers":     NUM_SPEAKERS,
    "speaker_map":      name_map,
    "speaker_label_map": mapping,
    "speaker_segments": mapped_hyp,
    "whisper_segments": result["segments"],
    "der_report": {
        "der_percent":       round(der,    2),
        "target":            "< 20%",
        "status":            "PASS" if der < 20 else "FAIL",
        "missed_speech_pct": round(ms_pct, 2),
        "false_alarm_pct":   round(fa_pct, 2),
        "confusion_pct":     round(sc_pct, 2),
        "total_ref_speech":  round(ref_s,  2),
    }
}

with open("/content/transcript.json", "w") as f:
    json.dump(output, f, indent=2)

with open("/content/transcript.txt", "w") as f:
    f.write(f"AUDIO: {os.path.basename(AUDIO_PATH)} | "
            f"DURATION: {int(duration//60)}m{int(duration%60)}s\n")
    f.write(f"SPEAKERS: {NUM_SPEAKERS} | "
            f"DER: {der:.2f}% | "
            f"Status: {'PASS' if der<20 else 'FAIL'}\n")
    f.write(f"MS={ms_pct:.2f}%  "
            f"FA={fa_pct:.2f}%  "
            f"SC={sc_pct:.2f}%\n")
    f.write("="*55 + "\n")
    f.write("\n".join(transcript_lines))
    f.write(f"\n{'='*55}\n")
    f.write(f"DER: {der:.2f}% | "
            f"{'PASS' if der<20 else 'FAIL'}\n")

print(f"\n🎉 Final DER: {der:.2f}% — "
      f"{'✅ TARGET MET!' if der < 20 else '❌ FAIL'}")



🔍 Loading pipeline...
✅ Pipeline ready

🔍 Running diarization (num_speakers=2)...
✅ Detected 2 speakers, 9 turns

───────────────────────────────────────────────────────
  #      Start     End  Speaker (raw)
───────────────────────────────────────────────────────
  1      0.03s  21.70s  SPEAKER_01
  2     11.57s  11.74s  SPEAKER_00
  3     19.30s  19.44s  SPEAKER_00
  4     21.70s  24.96s  SPEAKER_00
  5     25.11s  25.58s  SPEAKER_01
  6     26.37s  28.80s  SPEAKER_00
  7     30.24s  32.97s  SPEAKER_00
  8     30.46s  30.52s  SPEAKER_01
  9     33.63s  46.07s  SPEAKER_01
───────────────────────────────────────────────────────

  Label map: {'SPEAKER_01': 'speaker_1', 'SPEAKER_00': 'speaker_2'}


📋 SPEAKER-WISE TRANSCRIPT

speaker1:
  Well, it's the kickoff meeting for our project.
  And this is just what we're going to be doing over the next 25 minutes.
  So, first of all, just to kind of make sure that we all know each other.
  I'm Laura and I'm the project manager.
  Great.
  Do you